In [1]:
import numpy as np

phi = 0.2
n_a = 10
n_v = 5

In [2]:
def _insert_prob_distr(position: int, phi: float) -> np.ndarray:
    distribution = np.zeros(position + 1)
    for j in range(position + 1):
        distribution[j] = phi ** (position - j)
    return distribution / distribution.sum()


_insert_prob_distr(5, 0.9)

array([0.12602255, 0.14002506, 0.1555834 , 0.17287044, 0.19207827,
       0.21342029])

In [3]:
n_a = 20
n_v = 5
phi = 0.2

max_borda_score = (n_a - 1) * n_v

total_prob_0 = 0

for i in range(1, n_a):
    total_prob_0 += _insert_prob_distr(i, phi)[0]

print(f"Total prob of inserting at p0: {total_prob_0}")
print(f"Max borda score: {max_borda_score}")

Total prob of inserting at p0: 0.20693541439187935
Max borda score: 95


In [4]:
n_a_list = [5, 10, 15, 20, 40, 60, 80, 100]

phi = 0.9

for n_a in n_a_list:
    total_prob_0 = 0

    for i in range(1, n_a):
        total_prob_0 += _insert_prob_distr(i, phi)[0]

    print(f"n_a == {n_a}\t->\tProb p0: {total_prob_0:.3f}")

n_a == 5	->	Prob p0: 1.145
n_a == 10	->	Prob p0: 1.586
n_a == 15	->	Prob p0: 1.781
n_a == 20	->	Prob p0: 1.880
n_a == 40	->	Prob p0: 1.995
n_a == 60	->	Prob p0: 2.008
n_a == 80	->	Prob p0: 2.009
n_a == 100	->	Prob p0: 2.010


In [5]:
n_a_list = [5, 10, 15, 20, 40, 60, 80, 100]

phi = 0.7

for n_a in n_a_list:
    prob_1_p0 = _insert_prob_distr(1, phi)[0]

    print(f"n_a == {n_a}\t->\tProb p0: {prob_1_p0:.3f}")

n_a == 5	->	Prob p0: 0.412
n_a == 10	->	Prob p0: 0.412
n_a == 15	->	Prob p0: 0.412
n_a == 20	->	Prob p0: 0.412
n_a == 40	->	Prob p0: 0.412
n_a == 60	->	Prob p0: 0.412
n_a == 80	->	Prob p0: 0.412
n_a == 100	->	Prob p0: 0.412


In [6]:
import pandas as pd

# probabilities = pd.DataFrame(columns=["n_a", "n_v", "max_borda_score", "phi", "p1", "p2", "p3", "p4"])

# n_a_list = [5, 10, 15, 20]
n_a_list = [10, 20, 30, 40, 50]

n_v_list = [10, 50, 100, 200, 500, 1000]

phi_list = [0.1, 0.4, 0.7, 0.75, 0.80, 0.85, 0.90, 0.95, 1.0]

data_dict = {"n_a": [], "n_v": [], "max_borda_score": [], "phi": [], "p1": [], "p2": [], "p3": [], "p4": []}

for n_a in n_a_list:
    for n_v in n_v_list:
        max_borda_score = (n_a - 1) * n_v

        for phi in phi_list:
            data_dict["n_a"].append(n_a)
            data_dict["n_v"].append(n_v)
            data_dict["max_borda_score"].append(max_borda_score)
            data_dict["phi"].append(phi)
            data_dict["p1"].append(_insert_prob_distr(1, phi)[0])
            data_dict["p2"].append(_insert_prob_distr(2, phi)[0])
            data_dict["p3"].append(_insert_prob_distr(3, phi)[0])
            data_dict["p4"].append(_insert_prob_distr(4, phi)[0])

probs = pd.DataFrame(data_dict)

In [7]:
probs[(probs.n_v == 1000) & (probs.phi == 0.9)]

,n_a,n_v,max_borda_score,phi,p1,p2,p3,p4
51,10,1000,9000,0.9,0.473684,0.298893,0.21198,0.160216
105,20,1000,19000,0.9,0.473684,0.298893,0.21198,0.160216
159,30,1000,29000,0.9,0.473684,0.298893,0.21198,0.160216
213,40,1000,39000,0.9,0.473684,0.298893,0.21198,0.160216
267,50,1000,49000,0.9,0.473684,0.298893,0.21198,0.160216


In [16]:
def _mallows_vote(
    n_a: int,
    insert_distributions: list[np.ndarray],
    show_text: bool = True,
) -> np.ndarray:
    vote = np.zeros(1, dtype=int)
    if show_text:
        print(f"\nInitial vote: {vote} \n")
    for j in range(1, n_a):
        insert_distribution = insert_distributions[j - 1]
        if show_text:
            print(f"-{j}- insert_distribution: {insert_distribution}")
        index = np.random.choice(range(len(insert_distribution)), p=insert_distribution)
        if show_text:
            print(f"-{j}- The index is: {index}")
        vote = np.insert(vote, index, j)
        if show_text:
            print(f"Current vote: {vote} \n")
    return vote

In [37]:
phi = 0.8
n_v = 1000

n_a_list = [20, 30, 40, 50, 100, 1000]

for n_a in n_a_list:
    max_borda_score = (n_a - 1) * n_v

    insert_distributions = [_insert_prob_distr(i, phi) for i in range(1, n_a)]

    borda_count_n0 = 0

    for i in range(n_v):
        vote = _mallows_vote(n_a, insert_distributions, False)

        borda_count_n0 += n_a - np.where(vote == 0)[0][0]

    borda_count_norm_n0 = borda_count_n0 / max_borda_score

    print(f"Borda score norm for n_a == {n_a}: {borda_count_norm_n0:.3f}")

Borda score norm for n_a == 20: 0.853
Borda score norm for n_a == 30: 0.895
Borda score norm for n_a == 40: 0.927
Borda score norm for n_a == 50: 0.938
Borda score norm for n_a == 100: 0.971
Borda score norm for n_a == 1000: 0.997


In [13]:
from preflibtools.instances import OrdinalInstance
from preflibtools.instances.sampling import *
from ranking_aggregation.preflib_compatibility.preflib_utils import *

n_a = 900
n_v = 1000
phi = 0.8

instance = OrdinalInstance()

instance.populate_mallows(n_v, n_a, [1], [phi], [tuple(np.arange(n_a))])

# From the instance generated by preflib, get two Numpy arrays
# 1) a matrix where each row is a different ranking in the profile
# 2) the number of voters that chose that ranking
ranking, weights = preflib_divide(instance)

In [14]:
ranking.shape

(1000, 900)

In [15]:
weights.shape

(1000,)